In [ ]:
abfs_path="abfss://b0cceb94-d226-4dbd-a75b-789c5defa0a2@onelake.dfs.fabric.microsoft.com/f5ac5edb-ff4b-4a7b-ab7c-1b4182e0d30b/Files/Landing/crime_data.csv"

In [ ]:
df=spark.read.csv(path=abfs_path,header=True,inferSchema=True)
display(df)

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

v_schema = StructType([
    StructField('REF_DATE', IntegerType(), True),      # Year (e.g., 1998)
    StructField('GEO', StringType(), True),            # Geographic area (e.g., Canada)
    StructField('DGUID', StringType(), True),          # Data GUID
    StructField('Violations', StringType(), True),     # Description of violation
    StructField('Statistics', StringType(), True),     # Type of statistic (e.g., Actual incidents)
    StructField('UOM', StringType(), True),            # Unit of Measure (e.g., Number)
    StructField('UOM_ID', IntegerType(), True),        # Unit ID (e.g., 223)
    StructField('SCALAR_FACTOR', StringType(), True),  # e.g., 'units'
    StructField('SCALAR_ID', IntegerType(), True),     # e.g., 0
    StructField('VECTOR', StringType(), True),         # e.g., 'v44348247'
    StructField('COORDINATE', StringType(), True),     # e.g., '1.1.1'
    StructField('VALUE', DoubleType(), True),          # e.g., 2688540
    StructField('STATUS', StringType(), True),         # Status info (blank in sample)
    StructField('SYMBOL', StringType(), True),         # Symbol info (blank in sample)
    StructField('TERMINATED', StringType(), True),     # Termination info (blank)
    StructField('DECIMALS', IntegerType(), True)       # e.g., 0
])

In [ ]:
df=spark.read.format('csv').option('header','True').schema(v_schema).load(abfs_path)

In [ ]:
df.createOrReplaceTempView('t_new_data')

In [ ]:
%%sql
SELECT * from t_new_data

In [ ]:
Fabric_tblsales_silver="abfss://b0cceb94-d226-4dbd-a75b-789c5defa0a2@onelake.dfs.fabric.microsoft.com/f5ac5edb-ff4b-4a7b-ab7c-1b4182e0d30b/Tables/tblsales_bronze"
try:
    spark.read.format('delta').load(Fabric_tblsales_silver).createOrReplaceTempView('t_tblsales_bronze')
except:
    v_create_table=f"""CREATE TABLE IF NOT EXISTS tblsales_bronze (
    REF_DATE INT,
    GEO STRING,
    DGUID STRING,
    Violations STRING,
    Statistics STRING,
    UOM STRING,
    UOM_ID INT,
    SCALAR_FACTOR STRING,
    SCALAR_ID INT,
    VECTOR STRING,
    COORDINATE STRING,
    VALUE DOUBLE,
    STATUS STRING,
    SYMBOL STRING,
    TERMINATED STRING,
    DECIMALS INT
)
USING DELTA;"""
spark.sql(v_create_table)
    spark.read.format('delta').load(Fabric_tblsales_silver).createOrReplaceTempView('t_tblsales_bronze')

In [ ]:
sql_statement = f'''
MERGE INTO tblsales_bronze AS target
USING t_bronze_new_data AS source
ON  target.DGUID = source.DGUID
    AND target.REF_DATE = source.REF_DATE
    AND target.Violations = source.Violations

WHEN MATCHED THEN 
    UPDATE SET
        target.GEO = source.GEO,
        target.Statistics = source.Statistics,
        target.UOM = source.UOM,
        target.UOM_ID = source.UOM_ID,
        target.SCALAR_FACTOR = source.SCALAR_FACTOR,
        target.SCALAR_ID = source.SCALAR_ID,
        target.VECTOR = source.VECTOR,
        target.COORDINATE = source.COORDINATE,
        target.VALUE = source.VALUE,
        target.STATUS = source.STATUS,
        target.SYMBOL = source.SYMBOL,
        target.TERMINATED = source.TERMINATED,
        target.DECIMALS = source.DECIMALS

WHEN NOT MATCHED THEN 
    INSERT (
        REF_DATE,
        GEO,
        DGUID,
        Violations,
        Statistics,
        UOM,
        UOM_ID,
        SCALAR_FACTOR,
        SCALAR_ID,
        VECTOR,
        COORDINATE,
        VALUE,
        STATUS,
        SYMBOL,
        TERMINATED,
        DECIMALS
    )
    VALUES (
        source.REF_DATE,
        source.GEO,
        source.DGUID,
        source.Violations,
        source.Statistics,
        source.UOM,
        source.UOM_ID,
        source.SCALAR_FACTOR,
        source.SCALAR_ID,
        source.VECTOR,
        source.COORDINATE,
        source.VALUE,
        source.STATUS,
        source.SYMBOL,
        source.TERMINATED,
        source.DECIMALS
    );
'''
spark.sql(sql_statement).show()